# Scratch-разведка: recsys-hm

Черновой ноутбук для первого знакомства с сырыми данными.
Цель — НЕ финальный анализ, а быстро понять: что за колонки, какие типы,
где пропуски, как связаны таблицы. Результаты разведки потом фиксируются
в `src/data/loader.py` (схема типов) и используются в `01_eda_temporal.ipynb`
(содержательный EDA под Фазу 2).

Этот ноутбук можно не коммитить / не хранить долго — он одноразовый.


In [1]:
import polars as pl
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")  # путь относительно notebooks/
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Первый взгляд на сырые файлы

Читаем только первые N строк — не грузим файл целиком.

In [11]:
# articles.csv
articles_sample = pd.read_csv(RAW_DIR / "articles.csv", nrows=1000)
articles_sample.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [12]:
# customers.csv
customers_sample = pd.read_csv(RAW_DIR / "customers.csv", nrows=1000)
customers_sample.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [13]:
# transactions_train.csv
transactions_sample = pd.read_csv(RAW_DIR / "transactions_train.csv", nrows=1000)
transactions_sample.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


## 2. Паспорт датасета: shape, dtypes, память

Теперь читаем файлы целиком (или через Polars lazy, если тяжело) и смотрим на общую картину.

In [14]:
# Размеры файлов на диске
for f in ["articles.csv", "customers.csv", "transactions_train.csv"]:
    path = RAW_DIR / f
    size_mb = path.stat().st_size / 1024**2
    print(f"{f}: {size_mb:.1f} MB")

articles.csv: 34.5 MB
customers.csv: 197.5 MB
transactions_train.csv: 3326.4 MB


In [15]:
# articles — небольшой файл, можно полностью в pandas
articles_full = pd.read_csv(RAW_DIR / "articles.csv")
print("shape:", articles_full.shape)
articles_full.info(memory_usage="deep")

shape: (105542, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 n

In [16]:
# customers — тоже нормально в pandas
customers_full = pd.read_csv(RAW_DIR / "customers.csv")
print("shape:", customers_full.shape)
customers_full.info(memory_usage="deep")

shape: (1371980, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371980 entries, 0 to 1371979
Data columns (total 7 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   customer_id             1371980 non-null  object 
 1   FN                      476930 non-null   float64
 2   Active                  464404 non-null   float64
 3   club_member_status      1365918 non-null  object 
 4   fashion_news_frequency  1355969 non-null  object 
 5   age                     1356119 non-null  float64
 6   postal_code             1371980 non-null  object 
dtypes: float64(3), object(4)
memory usage: 512.3 MB


In [17]:
# transactions_train — большой файл (~600MB), лучше через Polars lazy,
# чтобы не грузить сразу всё в pandas
transactions_lazy = pl.scan_csv(RAW_DIR / "transactions_train.csv")

# .collect() всё равно материализует данные, но Polars сделает это быстрее
# и компактнее, чем pandas.read_csv на дефолтных типах
transactions_full = transactions_lazy.collect()
print("shape:", transactions_full.shape)
print(transactions_full.schema)
print(f"память: {transactions_full.estimated_size('mb'):.1f} MB")

shape: (31788324, 5)
OrderedDict([('t_dat', String), ('customer_id', String), ('article_id', Int64), ('price', Float64), ('sales_channel_id', Int64)])
память: 2970.9 MB


## 3. Пропуски (NaN / null)

Для каждой таблицы — сколько пропусков в каждой колонке и в каком проценте строк.

In [18]:
na_articles = articles_full.isnull().sum()
na_articles[na_articles > 0].sort_values(ascending=False)

detail_desc    416
dtype: int64

In [19]:
na_customers = customers_full.isnull().sum()
pct = (na_customers / len(customers_full) * 100).round(1)
pd.DataFrame({"n_missing": na_customers, "pct_missing": pct}).query("n_missing > 0")

,n_missing,pct_missing
FN,895050,65.2
Active,907576,66.2
club_member_status,6062,0.4
fashion_news_frequency,16011,1.2
age,15861,1.2


In [20]:
na_transactions = transactions_full.null_count()
na_transactions

t_dat,customer_id,article_id,price,sales_channel_id
u32,u32,u32,u32,u32
0,0,0,0,0


## 4. Природа каждой колонки

Для непонятных / неочевидных колонок — nunique() и value_counts(), чтобы понять,
что это: id, категория, число, дата, свободный текст.

In [21]:
# articles: сколько уникальных значений в каждой колонке
articles_full.nunique().sort_values()

index_group_no                       5
index_group_name                     5
perceived_colour_value_id            8
perceived_colour_value_name          8
index_name                          10
index_code                          10
product_group_name                  19
perceived_colour_master_name        20
perceived_colour_master_id          20
garment_group_name                  21
garment_group_no                    21
graphical_appearance_name           30
graphical_appearance_no             30
colour_group_name                   50
colour_group_code                   50
section_name                        56
section_no                          57
product_type_name                  131
product_type_no                    132
department_name                    250
department_no                      299
detail_desc                      43404
prod_name                        45875
product_code                     47224
article_id                      105542
dtype: int64

In [22]:
# Пример: смотрим на конкретную "непонятную" колонку подробнее
# (замени на любую, которая вызывает вопросы)
col = "product_type_name"
print(articles_full[col].value_counts().head(15))
print("...")
print(f"всего уникальных: {articles_full[col].nunique()}")

product_type_name
Trousers            11169
Dress               10362
Sweater              9302
T-shirt              7904
Top                  4155
Blouse               3979
Jacket               3940
Shorts               3939
Shirt                3405
Vest top             2991
Underwear bottom     2748
Skirt                2696
Hoodie               2356
Bra                  2212
Socks                1889
Name: count, dtype: int64
...
всего уникальных: 131


In [23]:
# customers: то же самое
customers_full.nunique().sort_values()

FN                              1
Active                          1
club_member_status              3
fashion_news_frequency          3
age                            84
postal_code                352899
customer_id               1371980
dtype: int64

In [24]:
col = "club_member_status"
print(customers_full[col].value_counts(dropna=False))

club_member_status
ACTIVE        1272491
PRE-CREATE      92960
NaN              6062
LEFT CLUB         467
Name: count, dtype: int64


In [25]:
col = "age"
print(customers_full[col].describe())
print("min/max:", customers_full[col].min(), customers_full[col].max())

count    1.356119e+06
mean     3.638696e+01
std      1.431363e+01
min      1.600000e+01
25%      2.400000e+01
50%      3.200000e+01
75%      4.900000e+01
max      9.900000e+01
Name: age, dtype: float64
min/max: 16.0 99.0


In [26]:
# transactions: кардинальность ключевых колонок
transactions_full.select([
    pl.col("customer_id").n_unique().alias("n_unique_customers"),
    pl.col("article_id").n_unique().alias("n_unique_articles"),
    pl.col("sales_channel_id").n_unique().alias("n_unique_channels"),
])

n_unique_customers,n_unique_articles,n_unique_channels
u32,u32,u32
1362281,104547,2


In [27]:
transactions_full["sales_channel_id"].value_counts()

sales_channel_id,count
i64,u32
1,9408462
2,22379862


In [28]:
# Диапазон дат — важно для будущего temporal split (Фаза 2)
transactions_full.select([
    pl.col("t_dat").min().alias("date_min"),
    pl.col("t_dat").max().alias("date_max"),
])

date_min,date_max
str,str
"""2018-09-20""","""2020-09-22"""


## 5. Ключи и связи между таблицами

Проверяем: действительно ли `article_id` / `customer_id` в transactions
полностью покрываются справочниками `articles` / `customers`,
или есть расхождения (значения, которых нет в одной из таблиц).

In [29]:
# customer_id: пересечение transactions <-> customers
customers_ids = set(customers_full["customer_id"])
transactions_customer_ids = set(transactions_full["customer_id"].unique().to_list())

missing_customers = transactions_customer_ids - customers_ids
print(f"customer_id из transactions, которых нет в customers: {len(missing_customers)}")
print(f"всего уникальных customer_id в transactions: {len(transactions_customer_ids)}")

customer_id из transactions, которых нет в customers: 0
всего уникальных customer_id в transactions: 1362281


In [30]:
# article_id: пересечение transactions <-> articles
articles_ids = set(articles_full["article_id"])
transactions_article_ids = set(transactions_full["article_id"].unique().to_list())

missing_articles = transactions_article_ids - articles_ids
print(f"article_id из transactions, которых нет в articles: {len(missing_articles)}")
print(f"всего уникальных article_id в transactions: {len(transactions_article_ids)}")

article_id из transactions, которых нет в articles: 0
всего уникальных article_id в transactions: 104547


## 6. Целостность и аномалии

Дубликаты, странные диапазоны значений, нарушения ожидаемой логики.

In [31]:
print("Дубликаты строк:")
print("  articles:", articles_full.duplicated().sum())
print("  customers:", customers_full.duplicated().sum())
print("  transactions:", transactions_full.is_duplicated().sum())

Дубликаты строк:
  articles: 0
  customers: 0
  transactions: 5518813


In [32]:
# Числовые диапазоны — ищем выбросы (отрицательная цена, нереальный возраст и т.п.)
print("price:")
print(transactions_full["price"].describe())
print()
print("age:")
print(customers_full["age"].describe())

price:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 3.1788324e7 │
│ null_count ┆ 0.0         │
│ mean       ┆ 0.027829    │
│ std        ┆ 0.019181    │
│ min        ┆ 0.000017    │
│ 25%        ┆ 0.015814    │
│ 50%        ┆ 0.025407    │
│ 75%        ┆ 0.033881    │
│ max        ┆ 0.591525    │
└────────────┴─────────────┘

age:
count    1.356119e+06
mean     3.638696e+01
std      1.431363e+01
min      1.600000e+01
25%      2.400000e+01
50%      3.200000e+01
75%      4.900000e+01
max      9.900000e+01
Name: age, dtype: float64


In [33]:
# Отрицательные / нулевые цены — это подозрительно для магазина
n_bad_price = transactions_full.filter(pl.col("price") <= 0).height
print(f"строк с price <= 0: {n_bad_price}")

строк с price <= 0: 0


In [34]:
# Полные дубликаты транзакций (один и тот же customer+article+date+price+channel)
# — если высокая доля, стоит понять, это реальные повторные покупки в один день
# или технический дубль строк
dup_transactions = transactions_full.is_duplicated().sum()
print(f"дублирующихся строк в transactions: {dup_transactions} "
      f"({dup_transactions / transactions_full.height * 100:.3f}%)")

дублирующихся строк в transactions: 5518813 (17.361%)


In [35]:
# Смотрим на конкретный пример дубля своими глазами
dup_mask = transactions_full.is_duplicated()
sample_dups = transactions_full.filter(dup_mask).head(10)
sample_dups

t_dat,customer_id,article_id,price,sales_channel_id
str,str,i64,f64,i64
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",501820043,0.016932,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",501820043,0.016932,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",671505001,0.033881,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",671505001,0.033881,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",631848002,0.033881,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",631848002,0.033881,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",631848002,0.033881,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",631848002,0.033881,2
"""2018-09-20""","""000aa7f0dc06cd7174389e76c9e132…",676827002,0.042356,2


In [38]:
dup_counts = (
    transactions_full
    .group_by(["t_dat", "customer_id", "article_id", "price", "sales_channel_id"])
    .agg(pl.len().alias("n_repeats"))
    .filter(pl.col("n_repeats") > 1)
    .sort("n_repeats", descending=True)
)
print(dup_counts.height, "уникальных дублирующихся комбинаций")
dup_counts.head(10)

2543908 уникальных дублирующихся комбинаций


t_dat,customer_id,article_id,price,sales_channel_id,n_repeats
str,str,i64,f64,i64,u32
"""2018-10-14""","""d00063b94dcb1342869d4994844a27…",678342001,0.006763,1,569
"""2019-02-16""","""94665b46e194622ccdbcadc0170f13…",629420001,0.008458,2,199
"""2019-02-11""","""8f5f1e993eff204ca7206cabe0fc6d…",189634001,0.013542,2,120
"""2019-10-16""","""0152964ef19824d631d28ee3327a01…",756322001,0.025407,2,120
"""2019-09-28""","""5cba04ed9a3759bc02a8a9e01efccc…",688558002,0.013542,2,117
"""2020-02-07""","""c163adbcac6e42e600224be5e1682a…",689365047,0.025407,2,114
"""2019-02-23""","""61da44a2758206d5701771f4315637…",507909001,0.021593,1,105
"""2018-09-24""","""a81e0b7657a090198d8138c95fae7d…",618480001,0.033881,2,100
"""2019-07-16""","""18d346ffea696df26aa1d53ee19afb…",719348003,0.022017,2,100


In [39]:
dup_counts["n_repeats"].value_counts().sort("n_repeats")

n_repeats,count
u32,u32
2,2285073
3,177237
4,55412
5,9259
6,8181
…,…
114,1
117,1
120,2


In [40]:
# Кто эти customer_id с экстремальными n_repeats — сколько всего у них транзакций
top_outlier_customer = "d00063b94dcb1342869d4994844a27"  # первые символы из твоей таблицы, нужно подставить полный id

# Найдём полный customer_id по частичному совпадению
outlier_full_id = (
    transactions_full
    .filter(pl.col("t_dat") == "2018-10-14")
    .filter(pl.col("article_id") == 678342001)
    .select("customer_id")
    .unique()
)
print(outlier_full_id)

shape: (7, 1)
┌─────────────────────────────────┐
│ customer_id                     │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ d00063b94dcb1342869d4994844a27… │
│ 06db4cd1e43fdd7b0030096b944efe… │
│ a62a7f6ef5142a744a9e3fc557f3b0… │
│ 3505031e99e112cf7478a4e1a7f4b6… │
│ 62fe54ebf54558db6752028bf0bb7c… │
│ 1124e7b1fb10a573d1aa6e9bc688ca… │
│ 2bc323df64847ed00c31d9283b8f4c… │
└─────────────────────────────────┘


In [41]:
# Сколько всего строк-транзакций у этого customer_id за всю историю
full_id = outlier_full_id["customer_id"][0]
n_total_transactions = transactions_full.filter(pl.col("customer_id") == full_id).height
print(f"Всего транзакций у этого customer_id: {n_total_transactions}")

Всего транзакций у этого customer_id: 814


In [42]:
# Основная масса: посмотрим отдельно распределение БЕЗ длинного хвоста
# (n_repeats от 2 до 10 — где лежит настоящая масса данных)
dup_counts.filter(pl.col("n_repeats") <= 10)["n_repeats"].value_counts().sort("n_repeats")

n_repeats,count
u32,u32
2,2285073
3,177237
4,55412
5,9259
6,8181
7,1733
8,2132
9,911
10,1112


In [43]:
# И отдельно — сколько строк "теряем" по объёму, а не по количеству уникальных комбинаций
# (это важнее для решения о дедупликации)
n_repeats_total = dup_counts.select((pl.col("n_repeats") - 1).sum()).item()
print(f"Лишних строк из-за дублей (сверх первого вхождения): {n_repeats_total}")
print(f"Это {n_repeats_total / transactions_full.height * 100:.2f}% от всех строк")

Лишних строк из-за дублей (сверх первого вхождения): 2974905
Это 9.36% от всех строк


## 7. Резюме находок

- customer_id — 64-символьный hex, уникален на покупателя, высокая кардинальность → Utf8,
  не Categorical. В loader.py перекодирован в customer_idx (Int64) через отдельный маппинг.
- postal_code — тоже hex-строка, высокая кардинальность → Utf8, не Categorical.
- age — есть пропуски (customers: 1 356 119 из 1 371 980 заполнено), диапазон 16-99,
  разумный. Хранить как nullable Float, не строгий Int8 без проверки на null.
- t_dat — диапазон дат: 2018-09-20 — 2020-09-22 (ровно 2 года). База для temporal split (Фаза 2).
- price — диапазон 0.000017–0.591525, нормализованная величина, отрицательных/нулевых нет.
- Расхождений между таблицами: 0 — все customer_id и article_id из transactions
  находят пару в customers/articles. loader.py отработал без warning.
- Полные дубликаты строк в transactions: 17.4% строк участвуют в дублях, но реальный
  "лишний объём" — 9.36% (2 974 905 строк сверх первого вхождения).
  - Основная масса дублей (n_repeats 2-10) — это, вероятно, обычное поведение
    покупателя (несколько единиц одного товара за один день). Составляет
    подавляющую долю уникальных дублирующихся комбинаций (2.28M из 2.54M — n_repeats=2).
  - Обнаружен явный выброс: customer_id с n_repeats=569 для одного товара за один день,
    у него 814 транзакций всего — не похоже на обычного розничного покупателя
    (вероятно, опт/корпоративный аккаунт). Похожие выбросы есть и с n_repeats
    199, 120, 117, 114, 105, 100...
  - Решение на Фазу 3: не удалять дубли слепо (несут сигнал силы предпочтения для ALS),
    но отфильтровать/пометить аномальных покупателей с экстремальным n_repeats
    (порог обсудить отдельно — например, топ-N по количеству транзакций или
    по max(n_repeats) на одну комбинацию).
